# MPNN — Neural Message Passing for Quantum Chemistry (2017)

_Papers · Graph Neural Networks_

**One framework swallows every graph net: a node sends MESSAGES along edges, AGGREGATES them, UPDATES itself; a READOUT then pools all nodes into one graph-level prediction.**

---

This notebook is a practice scaffold for the **MPNN — Neural Message Passing for Quantum Chemistry (2017)** lesson. We trace one message+aggregate+update step by hand, then build a full message → GRU update → sum readout network and train it on synthetic molecule-style graphs. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Trace one message pass by hand

Before any learning, we replay the lesson's worked example on a 3-node path `0–1–2`. With the simplest possible choices — message $M=h_w$ (just the neighbour's state) and update $U=h_v+m_v$ — one round is pure arithmetic.

Eq. 1 sums each node's neighbour states into a message, Eq. 2 adds that message back into the node, and Eq. 3 sums all node states into one graph-level number. The hub (node 1) grows fastest because it has two neighbours.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# --- Recompute the lesson's worked example: ONE message+aggregate+update step. ---
# 3-node path 0-1-2 (edges 0-1, 1-2). Simplest MPNN: message M = h_w, update U = h_v + m_v.
adj = [[1], [0, 2], [1]]                 # neighbour lists: node 0~{1}, node 1~{0,2}, node 2~{1}
h0  = torch.tensor([1., 2., 3.])         # initial 1-D node states
m1  = torch.tensor([sum(h0[w] for w in adj[v]) for v in range(3)])   # Eq.1: m_v = sum_{w in N(v)} h_w
h1  = h0 + m1                            # Eq.2 (additive update): h_v^1 = h_v^0 + m_v^1
yhat = h1.sum()                          # Eq.3 (sum readout): y = sum_v h_v^T
print("messages m^1     =", m1.tolist())     # [2.0, 4.0, 2.0]
print("updated states h^1=", h1.tolist())    # [3.0, 6.0, 5.0]
print("sum readout yhat  =", yhat.item())     # 14.0

### Step 2 — Generate synthetic molecule-style graphs

Each "molecule" is a small graph: a handful of atoms (nodes) with 4-dim features, randomly bonded, where each bond carries a type in $\{0,1,2\}$ (an edge feature).

The regression target is a stand-in "energy" that depends on the graph's *structure* — it grows with the bonds and their types. That structure-dependence is exactly what message passing must learn to read.

In [ ]:
# --- Tiny molecule-style graphs with a structure-dependent target. ---
# Each "molecule": n atoms (nodes), random bonds (edges) with a bond-type in {0,1,2} (edge feature).
# Target = a stand-in "energy": grows with how clustered the graph is (sum over bonds of bond type).
def make_molecule(n_atoms, seed):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n_atoms, 4, generator=g)        # 4-dim atom features
    edges, etypes = [], []
    y = 0.0
    for i in range(n_atoms):
        for j in range(i + 1, n_atoms):
            if torch.rand(1, generator=g).item() < 0.45:        # random bond
                et = int(torch.randint(0, 3, (1,), generator=g).item())   # bond type 0/1/2
                edges += [(i, j), (j, i)]                                  # undirected -> both ways
                etypes += [et, et]
                y += (et + 1)                                            # target depends on structure
    return x, edges, etypes, torch.tensor([float(y)])

dataset = [make_molecule(n_atoms=torch.randint(4, 8, (1,)).item(), seed=s) for s in range(200)]

### Step 3 — Build the generic MPNN module

Now the full network spelled out as Eqs. 1–3. The **message** is an edge network: each bond type indexes a learned $D\times D$ matrix that transforms the neighbour's state (Eq. 1, summed over neighbours).

The **update** is a `GRUCell` (as in Gated Graph Nets, Eq. 2), run for `T` rounds so information spreads `T` hops. The **readout** sums all node states — an order-invariant pooling — and passes them through an MLP (Eq. 3).

In [ ]:
# --- The generic MPNN: message (edge network) + GRU update + sum readout. ---
D = 16                                               # node hidden size

class MPNN(nn.Module):
    def __init__(self, n_etypes=3, T=3):
        super().__init__()
        self.T = T
        self.embed = nn.Linear(4, D)                 # h_v^0 from atom features
        # edge network: each bond type -> a D x D matrix A(e_vw)  (Eq.1 message M = A(e_vw) h_w)
        self.edge_mats = nn.Parameter(torch.randn(n_etypes, D, D) * (1.0 / D ** 0.5))
        self.gru = nn.GRUCell(D, D)                  # update U_t (Eq.2), as in Gated Graph NNs
        self.readout = nn.Sequential(nn.Linear(D, D), nn.ReLU(), nn.Linear(D, 1))  # R (Eq.3)

    def forward(self, x, edges, etypes):
        h = self.embed(x)                            # [n, D]
        for _ in range(self.T):                      # T rounds of message passing
            m = torch.zeros_like(h)
            for (i, j), et in zip(edges, etypes):    # Eq.1: m_i += A(e_ij) h_j  (summed over neighbours)
                m[i] = m[i] + self.edge_mats[et] @ h[j]
            h = self.gru(m, h)                        # Eq.2: h <- U(h, m)
        return self.readout(h.sum(0))                # Eq.3: y = MLP( sum_v h_v^T )  (sum is order-invariant)

### Step 4 — Train the graph-level regressor and evaluate

We split the 200 graphs into train/validation, then fit the 3-round MPNN with Adam to minimize mean-squared error on the energy target.

After training we report the validation MSE. (These are numbers from our small synthetic run, not the paper's QM9 benchmark.)

In [ ]:
# --- Train graph-level regression. ---
torch.manual_seed(0)
model = MPNN(T=3)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
train, val = dataset[:160], dataset[160:]

for epoch in range(40):
    model.train()
    for x, edges, etypes, y in train:
        opt.zero_grad()
        loss = F.mse_loss(model(x, edges, etypes), y)
        loss.backward()
        opt.step()

# --- Evaluate. ---
model.eval()
with torch.no_grad():
    val_mse = torch.stack([F.mse_loss(model(x, e, t), y) for x, e, t, y in val]).mean().item()
print(f"3-round MPNN  validation MSE = {val_mse:.3f}")
# (our small run on synthetic molecule-style graphs -- not the paper's QM9 numbers)

## Visualize the data & results

_Does the message-passing phase (Eqns. 1&ndash;2) actually let the model read graph structure? Compare a 3-round MPNN against the ablation that runs ZERO rounds (T=0 &mdash; readout straight off the raw node features)._

### Step 1 — Rebuild the data and a configurable MPNN

To run the ablation cleanly we rebuild the same synthetic graphs and an MPNN whose number of message-passing rounds `T` is a knob. When `T=0` the message-passing loop never runs, so the readout pools the raw node features directly.

Everything else — embedding, edge matrices, GRU update, readout — is identical, so any difference in error is attributable only to message passing.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Does the message-passing phase (Eqns. 1-2) let the model read graph structure?
# Compare a 3-round MPNN against the T=0 ablation (readout straight off raw features).
def make_molecule(n_atoms, seed):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n_atoms, 4, generator=g)
    edges, etypes, y = [], [], 0.0
    for i in range(n_atoms):
        for j in range(i+1, n_atoms):
            if torch.rand(1, generator=g).item() < 0.45:
                et = int(torch.randint(0, 3, (1,), generator=g).item())
                edges += [(i, j), (j, i)]
                etypes += [et, et]
                y += (et + 1)
    return x, edges, etypes, torch.tensor([float(y)])

data = [make_molecule(int(torch.randint(4, 8, (1,)).item()), s) for s in range(200)]
train, val = data[:160], data[160:]
D = 16

class MPNN(nn.Module):
    def __init__(self, T):
        super().__init__()
        self.T = T
        self.embed = nn.Linear(4, D)
        self.edge_mats = nn.Parameter(torch.randn(3, D, D) * (1.0 / D ** 0.5))
        self.gru = nn.GRUCell(D, D)
        self.readout = nn.Sequential(nn.Linear(D, D), nn.ReLU(), nn.Linear(D, 1))

    def forward(self, x, edges, etypes):
        h = self.embed(x)
        for _ in range(self.T):                       # T=0 -> this loop never runs (the ablation)
            m = torch.zeros_like(h)
            for (i, j), et in zip(edges, etypes):
                m[i] = m[i] + self.edge_mats[et] @ h[j]   # Eq.1: edge-network message, summed
            h = self.gru(m, h)                            # Eq.2: GRU update
        return self.readout(h.sum(0))                     # Eq.3: sum readout

### Step 2 — Run the ablation and compare learning curves

We train two models that differ in one knob only: `T=3` (passes messages) versus `T=0` (the ablation, readout off raw features). The `run` helper returns the per-epoch validation MSE curve.

The full MPNN should drive the error down sharply while the `T=0` ablation plateaus high — concrete evidence that message passing, not the readout or parameter count, is what reads graph structure.

In [ ]:
def run(T):
    torch.manual_seed(0)
    model = MPNN(T=T)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    curve = []
    for ep in range(40):
        model.train()
        for x, e, t, y in train:
            opt.zero_grad()
            F.mse_loss(model(x, e, t), y).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            curve.append(torch.stack([F.mse_loss(model(x, e, t), y) for x, e, t, y in val]).mean().item())
    return curve

mpnn = run(T=3)      # passes messages
abl  = run(T=0)      # ABLATION: zero rounds -> readout off raw features
idx = list(range(0, 40, 2))
print("MPNN (T=3):", [[i, round(mpnn[i], 2)] for i in idx])
print("Ablation (T=0):", [[i, round(abl[i], 2)] for i in idx])
# MPNN -> ~0.26;  ablation plateaus ~5.9  (our small run, not the paper's QM9 numbers)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have an MPNN ($T=3$ rounds of message+update, then a sum readout) that
            predicts a graph-level target which depends on how clustered the graph is. You set the number of
            message-passing rounds to $T=0$ (so the readout pools the raw input node features with
            no message passing) and retrain everything else identically. What happens to the regression error,
            and what does it isolate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change exactly one thing: set $T=0$ so Eqns. 1&ndash;2 never run; keep the same readout, optimizer, data, and epochs. — _An honest ablation changes only the message passing, so any difference in error is attributable to it &mdash; not to the readout or capacity._
- Retrain and compare validation mean-squared error: the $T=0$ model is much worse and plateaus high. — _With $T=0$ no node ever sees a neighbour, so the readout pools features that carry no structural information &mdash; the target depends on structure the model cannot observe._
- Conclude that the message-passing phase (Eqns. 1&ndash;2), not the readout or the parameter count, is what lets the model read graph structure. — _Both models have the same readout and comparable capacity; only the one that passes messages can encode neighbourhoods, isolating $M/U$ as the cause._

**Answer:** With $T=0$ the model never passes a single message, so each node's final state equals its raw
                 input and the sum readout pools structure-blind vectors &mdash; the validation error stays high
                 and flat. Restoring $T=3$ lets every node gather its neighbourhood over three hops, so the
                 readout sees structural information and the error drops sharply. Since the only change was the
                 number of message-passing rounds, this isolates the message-passing phase (Eqns.
                 1&ndash;2), not the readout or extra parameters, as the reason the MPNN can predict a
                 structure-dependent target. The CODEVIZ panel shows exactly this contrast.

</details>

**Problem 2.** For the 3-node path graph in the worked example (states $h^0=[1,2,3]$, edges $0\!-\!1$ and
            $1\!-\!2$), run a second message+update round with the same sum-of-neighbours message
            $M=h_w$ and additive update $U=h_v+m_v$. Compute $h^2$ and the sum readout $\hat{y}=\sum_v h_v^2$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Start from $h^1=[3,6,5]$ (the result of round 1). Message round 2: $m_0^2=h_1^1=6$; $m_1^2=h_0^1+h_2^1=3+5=8$; $m_2^2=h_1^1=6$. — _Eqn. 1 sums each node's current-round neighbour states &mdash; now using the updated $h^1$ values, not $h^0$._
- Update: $h_v^2=h_v^1+m_v^2$, so $h_0^2=3+6=9$, $h_1^2=6+8=14$, $h_2^2=5+6=11$. — _Eqn. 2 folds the new aggregated message into the previous state._
- Readout: $\hat{y}=9+14+11=34$. — _Eqn. 3 sums the final node states (the order-invariant sum readout)._

**Answer:** Round 2 gives messages $m^2=[6,8,6]$, states $h^2=[9,14,11]$, and sum readout
                 $\hat{y}=9+14+11=34$ (up from $14$ after round 1). The hub (node 1) again grows fastest because
                 it aggregates two neighbours each round &mdash; after two rounds it has also absorbed signal from
                 nodes 0 and 2 (its neighbours' neighbours), showing how $T$ rounds spread information
                 $T$ hops.

</details>

**Problem 3.** A colleague says "GCN, GraphSAGE, and GAT are three completely different graph networks, so they need
            three different implementations." Using the MPNN framework, explain why that is the wrong way to see
            it, and what actually differs between them.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the MPNN skeleton: every message-passing graph net is a choice of message $M$, update $U$, and readout $R$ run for $T$ rounds (Eqns. 1&ndash;3). — _The paper's whole point (&sect;2) is that prior models are instances of this one framework, not separate species._
- Identify what each fills in: GCN's $M$ is a fixed degree-normalized neighbour-average (no learned message net); GraphSAGE's $M$ samples neighbours then mean/max-pools; GAT's $M$ weights each neighbour by a learned attention score before summing. — _They differ only in the message/aggregation choice; the loop structure is identical._
- Conclude they share one implementation &mdash; a generic message-pass loop &mdash; with a pluggable $M$ (and $U$/$R$). — _This is exactly why libraries like PyTorch Geometric expose a single MessagePassing base class._

**Answer:** They are not three unrelated networks &mdash; they are three choices of the message
                 function $M$ inside the same MPNN skeleton (Eqns. 1&ndash;3). GCN uses a fixed
                 degree-normalized average, GraphSAGE samples neighbours and mean/max-pools them, and GAT weights
                 neighbours by learned attention &mdash; but all three run "aggregate messages (Eqn. 1), update
                 (Eqn. 2), repeat $T$ times, then read out (Eqn. 3)." So you implement one generic
                 message-passing loop and swap the $M$/$U$/$R$ blocks. That unification &mdash; turning a zoo of
                 architectures into one parameterized template &mdash; is the paper's central contribution.

</details>